<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment2/DNN_Assignment_2_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, datasets
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# Use smaller subset to avoid OOM
x_train, y_train = x_train[:1000], y_train[:1000]
x_test, y_test = x_test[:200], y_test[:200]

# Expand to 3 channels
x_train = np.stack([x_train]*3, axis=-1).astype("float32") / 255.0
x_test = np.stack([x_test]*3, axis=-1).astype("float32") / 255.0

# Resize in batches to save memory
def resize_batch(x):
    return tf.image.resize(x, [224, 224]).numpy()

x_train_resized = np.concatenate([resize_batch(x_train[i:i+500]) for i in range(0, len(x_train), 500)])
x_test_resized = np.concatenate([resize_batch(x_test[i:i+500]) for i in range(0, len(x_test), 500)])

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Activation functions
activations = ['relu', 'softmax', 'tanh', 'sigmoid']

# Choose one pretrained CNN
base_model = applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

results = {}

for act in activations:
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation=act),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train_resized, y_train, epochs=2, batch_size=32, verbose=0,
                        validation_data=(x_test_resized, y_test))

    acc = history.history['val_accuracy'][-1]
    results[act] = acc
    print(f"Activation: {act:8s} -> Validation Accuracy: {acc:.4f}")

print("\nSummary of activation performance on ResNet50:")
for act, acc in results.items():
    print(f"{act:<10}: {acc:.4f}")


Activation: relu     -> Validation Accuracy: 0.4100
Activation: softmax  -> Validation Accuracy: 0.1550
Activation: tanh     -> Validation Accuracy: 0.2850
Activation: sigmoid  -> Validation Accuracy: 0.2000

Summary of activation performance on ResNet50:
relu      : 0.4100
softmax   : 0.1550
tanh      : 0.2850
sigmoid   : 0.2000
